# `09_evaluation.ipynb` — Pure-Python Evaluation & Official Java Reconciliation (85–105 min CAS Session)

## 1. Pure-Python Evaluation Architecture (`src/germeval_llm/evaluation.py`)
All metrics in this repository are computed in **pure Python** (no Java required at runtime on Google Colab) across two complementary scoring modes:

1. **Strict Modern Set-Classification Metrics (`compute_set_classification_metrics`):**
   - Evaluates **every document** in the split, including the ~16–18% `relevance=false` documents whose gold set is empty (`$\emptyset$`).
   - If a model hallucinates `Allgemein:negative` on an irrelevant post, it is properly penalized with `FP += 1`.
   - Computes standard **Micro-Precision**, **Micro-Recall**, **Micro-F1** ($\frac{2\text{TP}}{2\text{TP} + \text{FP} + \text{FN}}$), **Macro-F1**, **Exact-Set Accuracy**, and **paired bootstrap 95% confidence intervals** ($B=1{,}000$ resamples).

2. **Exact Historical Java-Compatible Scorer (`evaluate_task_c_java_compatible`):**
   To guarantee 100% comparability with 2017–2018 GermEval papers without needing Java 8 on Colab, we decompiled `EvaluationScript.jar` and replicated its two historical quirks in pure Python:
   - **Quirk A — Empty-Gold Loop Skip (`GermevalEvaluator.java:186`):** The outer loop iterates over `for (String goldCat : goldCategories)`. When a document has 0 gold categories (`relevance=false`), the loop body never executes and any predicted labels on that document are silently ignored (`FP = 0`).
   - **Quirk B — The `"none"` Placeholder Jaccard Bug (`Fscore.java:37–69`):** When recording a false negative `(gold, "none")` or false positive `("none", pred)`, `getDistinctLabels(data)` includes `"none"` in the label set over which it sums `tp_sum`, `fp_sum`, and `fn_sum`. Because `"none"` never matches itself (`tp=0`), iterating over `"none"` adds all FNs to `fp_sum` and all FPs to `fn_sum`, **doubling `fp_sum + fn_sum`** and turning Java's `MICRO_F1` into the **Jaccard index**:
     $$\text{MICRO\_F1}_{\text{Java}} = \frac{2 \cdot \text{TP}}{2 \cdot \text{TP} + 2(\text{FP} + \text{FN})} = \frac{\text{TP}}{\text{TP} + \text{FP} + \text{FN}}$$

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
bench = pd.read_csv('results/metrics/benchmark.csv')
display(bench)

rec = json.loads(Path('results/metrics/official_java_reconciliation.json').read_text())
print('\nPure-Python Java-Compat Scorer vs. Official EvaluationScript.jar (6-decimal agreement):')
display(pd.DataFrame(rec['reconciliations']))

### Hands-on Demo: Running the Pure-Python Evaluator Directly on Any Prediction List
Below we run `evaluate_task_c_predictions()` in pure Python on a 3-document synthetic example so you can see both `strict` and `java_compat` metrics computed side by side:

In [ ]:
from germeval_llm.evaluation import evaluate_task_c_predictions
from germeval_llm.xml_parser import load_canonical_jsonl
from germeval_llm.inference import load_predictions_jsonl

dev_docs_5 = load_canonical_jsonl('data/canonical/dev.jsonl')[:5]
smoke_preds_5 = load_predictions_jsonl('results/predictions/smoke_lora_dev_dev.jsonl')
live_eval = evaluate_task_c_predictions(dev_docs_5, smoke_preds_5, n_bootstrap=200)
print('Pure-Python Evaluation Summary on 5-doc Smoke LoRA Run:')
print(json.dumps({
    'schema_validity_rate': live_eval['schema_validity_rate'],
    'valid_exact_rate': live_eval['valid_exact_rate'],
    'c1_strict_micro_f1': live_eval['task_c1_category_strict']['micro_f1'],
    'c1_java_compat_micro_f1': live_eval['task_c1_category_java_compat']['micro_f1'],
    'c2_strict_micro_f1': live_eval['task_c2_category_sentiment_strict']['micro_f1'],
    'c2_java_compat_micro_f1': live_eval['task_c2_category_sentiment_java_compat']['micro_f1'],
}, indent=2))

### Key Takeaways
- Our pure-Python evaluator (`evaluation.py`) requires **zero external Java dependencies** on Google Colab while reproducing both modern Strict Micro/Macro-F1 and exact 6-decimal `EvaluationScript.jar` scores.